In [21]:
"""
Automated ML Pipeline: House Price Prediction
Stages: Ingest → Preprocess → Train → Evaluate → Drift Detection
"""

import os, json, time, warnings
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from scipy.stats import ks_2samp, wasserstein_distance
import joblib

warnings.filterwarnings("ignore")

os.makedirs("artifacts", exist_ok=True)
os.makedirs("reports",   exist_ok=True)

PIPELINE_LOG = []

def log(stage, msg, data=None):
    entry = {"stage": stage, "time": datetime.now().isoformat(), "msg": msg}
    if data: entry["data"] = data
    PIPELINE_LOG.append(entry)
    print(f"  [{stage}] {msg}")

# ──────────────────────────────────────────────
# STAGE 1 · DATA INGESTION
# ──────────────────────────────────────────────
def _make_synthetic_housing(n=20640, seed=42):
    """Synthetic California-style housing dataset (no download needed)."""
    rng = np.random.default_rng(seed)
    lat  = rng.uniform(32.5, 42.0, n)
    lon  = rng.uniform(-124.5, -114.0, n)
    medinc       = rng.gamma(3, 1.5, n).clip(0.5, 15)
    houseage     = rng.uniform(1, 52, n)
    ave_rooms    = rng.gamma(5, 1.2, n).clip(1, 20)
    ave_bedrms   = (ave_rooms * rng.uniform(0.2, 0.4, n)).clip(0.5, 6)
    population   = rng.gamma(4, 300, n).clip(3, 35682)
    households   = (population * rng.uniform(0.2, 0.5, n)).clip(1, 6082)
    # Label: rough price model
    medhouseval  = (
        medinc * 0.4
        + (52 - houseage) * 0.01
        + ave_rooms * 0.05
        - np.abs(lat - 34) * 0.03
        + rng.normal(0, 0.3, n)
    ).clip(0.15, 5.0)
    return pd.DataFrame({
        "medinc": medinc, "houseage": houseage, "ave_rooms": ave_rooms,
        "ave_bedrms": ave_bedrms, "population": population,
        "households": households, "latitude": lat, "longitude": lon,
        "medhouseval": medhouseval,
    })

def ingest():
    print("\n━━━ STAGE 1 · DATA INGESTION ━━━")
    df = _make_synthetic_housing()

    n_before = len(df)
    # Inject realistic nulls (2 %)
    rng = np.random.default_rng(42)
    for col in ["ave_rooms", "ave_bedrms", "population"]:
        idx = rng.choice(df.index, size=int(0.02 * len(df)), replace=False)
        df.loc[idx, col] = np.nan

    log("INGEST", f"Loaded {n_before} rows × {df.shape[1]} cols")
    log("INGEST", f"Nulls injected → {df.isnull().sum().sum()} missing cells")
    log("INGEST", "Features", list(df.columns[:-1]))

    df.to_parquet("artifacts/raw_data.parquet", index=False)
    return df

# ──────────────────────────────────────────────
# STAGE 2 · PREPROCESSING
# ──────────────────────────────────────────────
def preprocess(df: pd.DataFrame):
    print("\n━━━ STAGE 2 · PREPROCESSING ━━━")
    target = "medinc" if "medinc" in df.columns else df.columns[-1]
    # Use MedHouseVal as the actual label
    target = "medhouseval"

    # --- median imputation for numeric nulls ---
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    for c in num_cols:
        if df[c].isnull().any():
            df[c].fillna(df[c].median(), inplace=True)
    log("PREPROCESS", "Null imputation complete (median strategy)")

    # --- feature engineering ---
    df["rooms_per_person"]   = df["ave_rooms"]   / (df["population"] + 1)
    df["bedrms_per_room"]    = df["ave_bedrms"]  / (df["ave_rooms"]  + 1)
    df["pop_density"]        = df["population"]  / (df["households"] + 1)
    df["income_location"]    = df["medinc"] * (1 / (np.abs(df["latitude"] - 36) + 1))
    log("PREPROCESS", "Feature engineering → 4 derived features added")

    # --- clip extreme outliers (1st/99th percentile) ---
    feat_cols = [c for c in df.columns if c != target]
    for c in feat_cols:
        lo, hi = df[c].quantile([0.01, 0.99])
        df[c]  = df[c].clip(lo, hi)
    log("PREPROCESS", "Outlier clipping applied (1-99 pct)")

    X = df.drop(columns=[target])
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    log("PREPROCESS", f"Train/Test split → {len(X_train)} / {len(X_test)}")

    X_train.to_parquet("artifacts/X_train.parquet", index=False)
    X_test.to_parquet("artifacts/X_test.parquet",  index=False)
    y_train.to_frame().to_parquet("artifacts/y_train.parquet", index=False)
    y_test.to_frame().to_parquet("artifacts/y_test.parquet",   index=False)

    return X_train, X_test, y_train, y_test

# ──────────────────────────────────────────────
# STAGE 3 · TRAINING (model comparison)
# ──────────────────────────────────────────────
def train(X_train, y_train):
    print("\n━━━ STAGE 3 · TRAINING ━━━")
    candidates = {
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=300, max_depth=4,
            learning_rate=0.05, subsample=0.8, random_state=42),
        "RandomForest": RandomForestRegressor(
            n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
        "Ridge": Ridge(alpha=1.0),
    }

    results = {}
    best_score, best_name, best_pipe = -np.inf, None, None

    for name, model in candidates.items():
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", model)])
        cv   = cross_val_score(pipe, X_train, y_train,
                               cv=5, scoring="r2", n_jobs=-1)
        pipe.fit(X_train, y_train)
        results[name] = {"cv_r2_mean": round(cv.mean(), 4),
                         "cv_r2_std":  round(cv.std(),  4)}
        log("TRAIN", f"{name}: CV R² = {cv.mean():.4f} ± {cv.std():.4f}")
        if cv.mean() > best_score:
            best_score, best_name, best_pipe = cv.mean(), name, pipe

    log("TRAIN", f"Best model → {best_name} (CV R²={best_score:.4f})")
    joblib.dump(best_pipe, "artifacts/best_model.pkl")

    with open("artifacts/training_results.json", "w") as f:
        json.dump({"candidates": results, "winner": best_name}, f, indent=2)

    return best_pipe, best_name, results

# ──────────────────────────────────────────────
# STAGE 4 · EVALUATION
# ──────────────────────────────────────────────
def evaluate(pipe, X_test, y_test):
    print("\n━━━ STAGE 4 · EVALUATION ━━━")
    y_pred  = pipe.predict(X_test)
    rmse    = np.sqrt(mean_squared_error(y_test, y_pred))
    mae     = mean_absolute_error(y_test, y_pred)
    r2      = r2_score(y_test, y_pred)
    mape    = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-9))) * 100

    metrics = {"RMSE": round(rmse,4), "MAE": round(mae,4),
               "R2":   round(r2,4),   "MAPE%": round(mape,2)}
    log("EVAL", "Test metrics", metrics)

    # residuals
    residuals = y_test.values - y_pred
    res_stats = {"mean": round(float(residuals.mean()),4),
                 "std":  round(float(residuals.std()),4),
                 "skew": round(float(pd.Series(residuals).skew()),4)}
    log("EVAL", "Residual stats", res_stats)

    with open("reports/evaluation.json", "w") as f:
        json.dump({"metrics": metrics, "residuals": res_stats}, f, indent=2)

    return metrics, y_pred

# ──────────────────────────────────────────────
# STAGE 5 · DATA DRIFT DETECTION
# ──────────────────────────────────────────────
def detect_drift(X_train, X_test):
    print("\n━━━ STAGE 5 · DRIFT DETECTION ━━━")

    # ── Simulate drift: shift test distribution ──
    X_drifted = X_test.copy()
    rng = np.random.default_rng(99)

    drift_config = {}
    for col in ["medinc", "ave_rooms", "population", "rooms_per_person"]:
        if col not in X_drifted.columns: continue
        shift = X_drifted[col].std() * 1.5          # 1.5σ shift
        noise = rng.normal(shift, X_drifted[col].std() * 0.3, size=len(X_drifted))
        X_drifted[col] = X_drifted[col] + noise
        drift_config[col] = {"shift_sigma": 1.5}

    log("DRIFT", f"Drift simulated on {list(drift_config.keys())}")

    # ── KS test + Wasserstein distance per feature ──
    drift_results = {}
    ALERT_THRESHOLD_KS = 0.15
    ALERT_THRESHOLD_WD = 0.3

    for col in X_train.columns:
        ref  = X_train[col].dropna().values
        cur  = X_drifted[col].dropna().values
        ks_stat, ks_pval = ks_2samp(ref, cur)
        wd = wasserstein_distance(ref / (ref.std() + 1e-9),
                                  cur  / (ref.std() + 1e-9))   # normalised

        drifted = (ks_stat > ALERT_THRESHOLD_KS) or (wd > ALERT_THRESHOLD_WD)
        drift_results[col] = {
            "ks_statistic":  round(float(ks_stat),4),
            "ks_pvalue":     round(float(ks_pval),6),
            "wasserstein":   round(float(wd),4),
            "drift_detected": bool(drifted),
        }

    drifted_cols = [c for c, v in drift_results.items() if v["drift_detected"]]
    log("DRIFT", f"Drifted features ({len(drifted_cols)}): {drifted_cols}")

    # PSI (Population Stability Index) for top features
    def psi(ref_arr, cur_arr, bins=10):
        eps = 1e-6
        lo, hi = min(ref_arr.min(), cur_arr.min()), max(ref_arr.max(), cur_arr.max())
        edges  = np.linspace(lo, hi, bins + 1)
        r_pct  = np.histogram(ref_arr, bins=edges)[0] / len(ref_arr) + eps
        c_pct  = np.histogram(cur_arr, bins=edges)[0] / len(cur_arr) + eps
        return float(np.sum((c_pct - r_pct) * np.log(c_pct / r_pct)))

    psi_results = {}
    for col in X_train.columns:
        p = psi(X_train[col].dropna().values, X_drifted[col].dropna().values)
        psi_results[col] = {
            "psi": round(p, 4),
            "severity": "CRITICAL" if p > 0.25 else "WARNING" if p > 0.1 else "OK"
        }
        log("DRIFT", f"  PSI [{col}] = {p:.4f} → {psi_results[col]['severity']}")

    report = {
        "drift_simulation": drift_config,
        "ks_wasserstein":   drift_results,
        "psi":              psi_results,
        "summary": {
            "features_monitored": len(X_train.columns),
            "drifted_count": len(drifted_cols),
            "drifted_features": drifted_cols,
            "critical_psi": [c for c, v in psi_results.items() if v["severity"] == "CRITICAL"],
        }
    }
    with open("reports/drift_report.json", "w") as f:
        json.dump(report, f, indent=2)

    return report

# ──────────────────────────────────────────────
# ORCHESTRATOR
# ──────────────────────────────────────────────
def run_pipeline():
    start = time.time()
    print("╔══════════════════════════════════════╗")
    print("║   AUTOMATED ML PIPELINE  v1.0        ║")
    print("║   House Price Prediction              ║")
    print("╚══════════════════════════════════════╝")

    df                      = ingest()
    X_train,X_test,y_train,y_test = preprocess(df)
    pipe, best_name, cv_res = train(X_train, y_train)
    metrics, y_pred         = evaluate(pipe, X_test, y_test)
    drift_report            = detect_drift(X_train, X_test)

    elapsed = round(time.time() - start, 2)

    summary = {
        "pipeline_version": "1.0",
        "run_at": datetime.now().isoformat(),
        "elapsed_seconds": elapsed,
        "best_model": best_name,
        "test_metrics": metrics,
        "drift_summary": drift_report["summary"],
        "log": PIPELINE_LOG,
    }
    with open("reports/pipeline_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n✅  Pipeline complete in {elapsed}s")
    print(f"    Best model : {best_name}")
    print(f"    Test R²    : {metrics['R2']}")
    print(f"    RMSE       : {metrics['RMSE']}")
    print(f"    Drifted    : {drift_report['summary']['drifted_features']}")
    return summary

if __name__ == "__main__":
    summary = run_pipeline()

╔══════════════════════════════════════╗
║   AUTOMATED ML PIPELINE  v1.0        ║
║   House Price Prediction              ║
╚══════════════════════════════════════╝

━━━ STAGE 1 · DATA INGESTION ━━━
  [INGEST] Loaded 20640 rows × 9 cols
  [INGEST] Nulls injected → 1236 missing cells
  [INGEST] Features

━━━ STAGE 2 · PREPROCESSING ━━━
  [PREPROCESS] Null imputation complete (median strategy)
  [PREPROCESS] Feature engineering → 4 derived features added
  [PREPROCESS] Outlier clipping applied (1-99 pct)
  [PREPROCESS] Train/Test split → 16512 / 4128

━━━ STAGE 3 · TRAINING ━━━
  [TRAIN] GradientBoosting: CV R² = 0.9172 ± 0.0024
  [TRAIN] RandomForest: CV R² = 0.9141 ± 0.0025
  [TRAIN] Ridge: CV R² = 0.9165 ± 0.0020
  [TRAIN] Best model → GradientBoosting (CV R²=0.9172)

━━━ STAGE 4 · EVALUATION ━━━
  [EVAL] Test metrics
  [EVAL] Residual stats

━━━ STAGE 5 · DRIFT DETECTION ━━━
  [DRIFT] Drift simulated on ['medinc', 'ave_rooms', 'population', 'rooms_per_person']
  [DRIFT] Drifted featu